Полезные ссылки
Урок в прозе: https://proproprogs.ru/python_oop/python-oop-vvedenie-v-python-data-classes-chast-2

Телеграм-канал: https://t.me/python_selfedu

Представим, что нам при инициализации требуется создать вычисляемое свойство

In [1]:
class Vector30:
    def init (self, x: int, y: int, z: int):
        self.x = x
        self.y = y
        self.z = z
        self.length = (x*x+y*y+z*z) ** 0.5

In [3]:
from dataclasses import dataclass, field

@dataclass
class V3D:
    x: int
    y: int
    z: int
# как же нам создать length??
# Инициализаторы дата-классов в конце своей работы вызывают метод post_init
    def __post_init__(self):
        self.length = (self.x*self.x + self.y*self.y + self.z*self.z) ** 0.5

v = V3D(1,2,3)
print(v)
print(v.__dict__)
# вычисляемое свойство есть только в __dict__, так как в метод __repr__ попадает только все, обхявленное до post_init


V3D(x=1, y=2, z=3)
{'x': 1, 'y': 2, 'z': 3, 'length': 3.7416573867739413}


Как же сделать так, чтобы length формировался в инициализаоре (вместе с остальными), но при этом его не было нужно передавать. Но прописать в post_init логику формирования length все же стоит

In [7]:
@dataclass
class V3D:
    x: int
    y: int
    z: int
    length: float = field(init=False)

    def __post_init__(self):
        self.length = (self.x*self.x + self.y*self.y + self.z*self.z) ** 0.5

v = V3D(1,2,3)
print(v)

V3D(x=1, y=2, z=3, length=3.7416573867739413)


В поле field кроме init и default_factory есть несколько других аргументов:
repr - нужно ли выводить атрибут при вызове repr
compare - нужно использовать атрибут при сравнении объектов
default - значение по умолчанию

In [12]:
@dataclass
class V3D:
    x: int = field(repr=False)
    y: int
    z: int = field(compare=False, default=0)
    length: float = field(init=False, compare=False)

    def __post_init__(self):
        self.length = (self.x*self.x + self.y*self.y + self.z*self.z) ** 0.5


v1 = V3D(1,2,3)
v2 = V3D(1,2,565656)
v3 = V3D(4,5)
print(v1)
print(v1==v2)
print(v3)

V3D(y=2, z=3, length=3.7416573867739413)
True
V3D(y=5, z=0, length=6.4031242374328485)


Теперь представим, что length мы бы хотели формировать по некоторому булеву флагу. В обычном классе это было бы написано вот так

In [ ]:
class Vector30:
    def init (self, x: int, y: int, z: int, calc_len: bool=True):
        self.x = x
        self.y = y
        self.z = z
        self.length = (x*x+y*y+z*z) ** 0.5 if calc_len else 0

In [18]:
from dataclasses import dataclass, field, InitVar

@dataclass
class V3D:
    x: int = field(repr=False)
    y: int
    z: int = field(compare=False)
    calc_len: InitVar[bool] = True # если какой-то парамер аннотирован через InitVar, то он автоматически передается в метод __post_init__
    length: float = field(init=False, compare=False, default=0) # условие флага calc_len здесь реализовано через default

    def __post_init__(self, calc_len: bool):
        if calc_len:
            self.length = (self.x*self.x + self.y*self.y + self.z*self.z) ** 0.5


print(V3D(1,2,3))
print(V3D(1,2,3, False))


V3D(y=2, z=3, length=3.7416573867739413)
V3D(y=2, z=3, length=0)


############################
Ранее декоратор @dataclass мы использовали лишь с параметрами по умолчанию, но там можно отключить ввод парметров через repr, отключить сравнение на ><=, убрать инициализатор (если предполагается, что это будет базовый класс для кого-то), при помощи frozen=True можно запретить менять атрибуты экземпляра извне